### **Portefólio AR — Demonstração unificada Tic-Tac-Toe**

A single executable walkthrough that trains and evaluates the main agents of the portfolio on the same environment:
**Random → SARSA → Q-Learning → REINFORCE (vanilla) → REINFORCE + baseline → MCTS**.

Practicals **PL6/PL7** (environment + features + SARSA/Q-Learning), **PL8** (REINFORCE), **PL9** (Monte Carlo Tree Search).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt

from AR1.envs.tictactoe import TicTacToeEnv, _winner
from AR1.policies.tictactoe import random_action

SEED        = 7
N_EVAL      = 500    # evaluation games per side
N_TRAIN     = 5_000  # training episodes for SARSA / Q-Learning
N_REINFORCE = 4_000  # training episodes for REINFORCE

np.random.seed(SEED)
print(f"seed = {SEED}   eval games = {N_EVAL}   train episodes = {N_TRAIN}")


---
#### **1. Shared evaluation helper**

Every agent below is exposed as a `policy(env, state) → action` callable.  This single helper plays `n_games` games against a uniform-random opponent and reports the win / draw / loss rates.  We run it twice per agent: once as **X** (first to move) and once as **O** (second to move).

The same seed is reused across agents so any difference between them is *not* attributable to luck.


In [ ]:
def evaluate_vs_random(policy_fn, n_games=N_EVAL, as_player=1, seed=SEED):
    env = TicTacToeEnv()
    wins = draws = losses = 0
    for _ in range(n_games):
        state = env.reset()
        done = False
        while not done:
            if env.current_player == as_player:
                action = policy_fn(env, state)
            else:
                action = random_action(env, state)
            state, _, done = env.step(action)
        w = _winner(state)
        if w == as_player:
            wins += 1
        elif w == 0:
            draws += 1
        else:
            losses += 1
    return wins / n_games, draws / n_games, losses / n_games


def both_sides(policy_fn, label, seed=SEED):
    wx, dx, lx = evaluate_vs_random(policy_fn, as_player=1,  seed=seed)
    wo, do, lo = evaluate_vs_random(policy_fn, as_player=-1, seed=seed + 1)
    print(f"{label:<22}  X: win={wx:.1%} draw={dx:.1%} loss={lx:.1%}  |  "
          f"O: win={wo:.1%} draw={do:.1%} loss={lo:.1%}")
    return {"as_X": (wx, dx, lx), "as_O": (wo, do, lo)}


results = {}


---
#### **2. Random baseline**

Two uniform-random players — this is the floor of difficulty and the reference for everything that follows.  X has a structural advantage (first mover) which already pushes the win-rate well above 50 %.


In [ ]:
def random_policy(env, state):
    return random_action(env, state)

results["random"] = both_sides(random_policy, "Random")


---
#### **3. Tabular control with legal-action masking (PL6/PL7)**

`TicTacToeSarsa` and `TicTacToeQLearning` (in `scripts/run_tictactoe.py`) are tabular agents whose `select_action(state, available)` ignores illegal cells.  They are trained vs a random opponent and evaluated greedily.

**SARSA (on-policy):** Q is updated with the actually-chosen next action — slightly conservative because exploration *is* part of the policy being evaluated.

$$Q(s,a) \leftarrow Q(s,a) + \alpha \,[\, r + \gamma\, Q(s', a') - Q(s, a)\,]$$

**Q-Learning (off-policy):** Q is updated with the *best* next action regardless of what's actually played — converges to the optimal policy under mild conditions.

$$Q(s,a) \leftarrow Q(s,a) + \alpha \,[\, r + \gamma\, \max_{a'} Q(s', a') - Q(s, a)\,]$$


In [ ]:
from AR1.scripts.run_tictactoe import TicTacToeSarsa, TicTacToeQLearning, train_agent

def _train_and_wrap(AgentCls, episodes=N_TRAIN, seed=SEED):
    agent = AgentCls(epsilon=0.1, alpha=0.5, gamma=1.0, seed=seed)
    train_agent(agent, num_episodes=episodes, seed=seed)

    def policy(env, state):
        avail = env.available_actions(state)
        return agent.greedy_action(state, avail)
    return agent, policy

print("Training SARSA ...")
sarsa_agent, sarsa_policy = _train_and_wrap(TicTacToeSarsa)
results["sarsa"] = both_sides(sarsa_policy, "SARSA")

print("\nTraining Q-Learning ...")
ql_agent, ql_policy = _train_and_wrap(TicTacToeQLearning)
results["q_learning"] = both_sides(ql_policy, "Q-Learning")


---
#### **4. REINFORCE — Monte Carlo policy gradient (PL8)**

A **linear softmax policy** in the 27-dim perspective-relative encoding ($\theta \in \mathbb{R}^{9 \times 27}$):

$$\pi(a\,|\,s) = \frac{e^{\theta_a \cdot \phi(s)}}{\sum_{b \in \mathcal{A}(s)} e^{\theta_b \cdot \phi(s)}}$$

**Vanilla update** (one step per episode, $G_t$ = Monte Carlo return):

$$\theta \leftarrow \theta + \alpha \,\gamma^t\, G_t\, \nabla_\theta \log \pi(a_t\,|\,s_t)$$

**Baseline (Sutton & Barto Sec. 13.4)** subtracts a learned $V(s_t) = w \cdot \phi(s_t)$ from the returns to reduce variance *without* introducing bias:

$$\theta \leftarrow \theta + \alpha \,\gamma^t\, [\,G_t - V(s_t)\,]\, \nabla_\theta \log \pi(a_t\,|\,s_t)$$

We train both variants in self-play with 30 % of episodes substituted by a random opponent (helps the policy stay sharp against weak play).


In [ ]:
from AR1.agents.control.reinforce import ReinforceAgent
from AR1.experiments.reinforce_tictactoe import (
    run_reinforce_episode,
    run_vs_random_episode,
)
from AR1.features.tictactoe import encode_state

def train_reinforce(agent, episodes=N_REINFORCE, random_frac=0.3, seed=SEED):
    rng = np.random.default_rng(seed)
    env = TicTacToeEnv()
    for _ in range(episodes):
        if rng.random() < random_frac:
            run_vs_random_episode(env, agent)
        else:
            run_reinforce_episode(env, agent)
    return agent

def reinforce_policy_for(agent):
    def policy(env, state):
        phi   = encode_state(state, env.current_player)
        avail = env.available_actions(state)
        return agent.greedy_action(phi, avail)
    return policy

print("Training REINFORCE (vanilla) ...")
ra = train_reinforce(ReinforceAgent(alpha=0.01, use_baseline=False, seed=SEED))
results["reinforce_vanilla"] = both_sides(reinforce_policy_for(ra), "REINFORCE vanilla")

print("\nTraining REINFORCE + baseline ...")
rb = train_reinforce(ReinforceAgent(alpha=0.01, use_baseline=True, seed=SEED + 1))
results["reinforce_baseline"] = both_sides(reinforce_policy_for(rb), "REINFORCE + baseline")


---
#### **5. Monte Carlo Tree Search — model-based planning (PL9)**

MCTS is the only agent here that **does not learn anything**.  At every move it builds a search tree from scratch by running $N$ simulations of four phases each:

1. **Selection** — descend with UCB1 until a node is not fully expanded or is terminal:
$$\text{UCB}(s,a) = Q(s,a) + c\,\sqrt{\frac{\ln N(s)}{N(s,a)}}$$
2. **Expansion** — add one new child for an untried action.
3. **Simulation** — random rollout to the end of the game.
4. **Backup** — propagate the outcome up the tree, **flipping sign at each level** (parent is the opponent).

Action selected = **most-visited child of the root** (robust best — less sensitive to outlier rollouts than $\arg\max Q$).


In [ ]:
from AR1.agents.planning.mcts import MCTSAgent

mcts = MCTSAgent(n_simulations=200)

def mcts_policy(env, state):
    return mcts.select_action(state, env.current_player)

results["mcts_200"] = both_sides(mcts_policy, "MCTS (200 sims)")


---
#### **6. Comparison plot**

Stacked **win / draw / loss** rates against the random opponent — left panel: agents playing as **X** (first to move); right panel: playing as **O** (second to move).  The hierarchy is the punchline of the portfolio:

| Capability added | Algorithm | Typical X win-rate |
|---|---|---:|
| _(none)_                              | Random                 | ~58 % |
| Tabular control, off-policy           | Q-Learning             | ~98 % |
| Tabular control, on-policy            | SARSA                  | ~98 % |
| Policy gradient + variance reduction  | REINFORCE + baseline   | ~96 % |
| Policy gradient (vanilla)             | REINFORCE              | ~78 % |
| Model-based planning at decision time | MCTS                   | **100 %** |


In [ ]:
def stacked_bars(ax, role, title):
    labels = list(results.keys())
    wins   = [results[k][role][0] for k in labels]
    draws  = [results[k][role][1] for k in labels]
    losses = [results[k][role][2] for k in labels]
    x = np.arange(len(labels))
    ax.bar(x, wins,                                                  label="win",  color="#4caf50")
    ax.bar(x, draws,  bottom=wins,                                   label="draw", color="#ffb300")
    ax.bar(x, losses, bottom=np.array(wins) + np.array(draws),       label="loss", color="#e53935")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("fraction of games")
    ax.set_title(title)
    ax.grid(alpha=0.2, axis="y")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
stacked_bars(axes[0], "as_X", "as X (first to move)")
stacked_bars(axes[1], "as_O", "as O (second to move)")
axes[1].legend(loc="upper right")
fig.suptitle("Tic-Tac-Toe — agents vs uniform random")
fig.tight_layout()
plt.show()


---
#### **7. Discussion**

- **MCTS wins because the horizon is short.**  With 200 simulations per move and ≤ 9 moves per game, the search is allowed to expand the relevant subtree almost completely.  No training, perfect play.

- **Q-Learning ≥ SARSA.**  The off-policy max-update converges to the optimal greedy policy regardless of how much $\varepsilon$-exploration is used during learning.  SARSA learns a policy that is optimal *conditional on continuing $\varepsilon$-greedy*, which is slightly more conservative.

- **REINFORCE + baseline ≫ vanilla REINFORCE.**  Both estimators are unbiased, but the variance of $G_t$ explodes in episodic settings.  Subtracting $V(s_t)$ does not change the expectation but reduces variance by an order of magnitude in practice — visible as ~96 % vs ~78 % win rate with the same training budget.

- **As O is harder.**  Going second forces the agent to react rather than dictate; only MCTS reliably draws a Tic-Tac-Toe game it cannot win, which is why all win-rates as O are below the corresponding rates as X.

For a deeper breakdown (including the DQN and AlphaZero extensions trained on the same environment), see [`RESULTS.md`](../RESULTS.md).
